# Guitar Audio DSP

Analysis and processing of a short acoustic-electric guitar recording with NumPy and SciPy.

I picked an acoustic-electric guitar when I started learning because I wanted one instrument that could cover both a natural acoustic tone and a driven electric sound. This project explores the second half of that idea digitally: start from a real recording of two chords I was practicing, look at its time and frequency structure, then use a nonlinear saturation stage and frequency-selective EQ to push it toward a more driven, electric-style tone.

The notebook walks through the same pipeline implemented in [`signal_processing.py`](../signal_processing.py) and calls its functions directly, so the two always describe the same algorithm. Outputs are cleared to keep the file small; the generated figures and audio are committed under `results/` and `audio/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

%matplotlib inline

# Make signal_processing.py importable whether the notebook runs from notebooks/ or the repo root
REPO_ROOT = Path.cwd() if (Path.cwd() / "signal_processing.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from signal_processing import (
    DRIVE, HEADROOM, load_audio, peak_normalize, compute_spectrum, find_dominant_peaks,
    plot_waveform, plot_spectrum, plot_spectrogram, soft_clip, apply_guitar_eq,
    generate_comparisons, save_audio,
)

SOURCE_WAV = REPO_ROOT / "audio" / "source_guitar.wav"
PROCESSED_WAV = REPO_ROOT / "audio" / "processed_guitar.wav"
RESULTS_DIR = REPO_ROOT / "results"

## Source recording

The clip was recorded directly into GarageBand and exported as a 16-bit stereo WAV at 44.1 kHz. It is mixed to mono by averaging the two channels and scaled to [-1, 1].

It contains two chords strummed repeatedly:

| Chord | Fretted notes |
|-------|---------------|
| G major | G2, B2, D4, G4 |
| Cadd9 | C3, E3, D4, G4 |

D4 and G4 are shared by both chords, so they are expected to dominate the long-window spectrum.

In [ ]:
fs, x = load_audio(SOURCE_WAV)
duration = len(x) / fs
t = np.arange(len(x)) / fs

print(f"Sampling rate : {fs} Hz")
print(f"Duration      : {duration:.2f} s")
print(f"Total samples : {len(x):,}")
display(Audio(x, rate=fs))

## Signal analysis

The signal is examined three ways:

1. **Time-domain waveform**: amplitude envelope, strum onsets, and decay.
2. **FFT magnitude and phase**: frequency content of the whole clip. `rfft` is used because the input is real, and the one-sided amplitude is doubled for every bin except DC and Nyquist.
3. **STFT spectrogram**: how that content evolves over time (2048-sample Hann window, 75% overlap).

In [ ]:
plot_waveform(t, x, RESULTS_DIR / "waveform.png");

In [ ]:
freqs, mag, phase = compute_spectrum(x, fs)
peaks = find_dominant_peaks(freqs, mag)

print("Dominant spectral peaks below 800 Hz:")
for f, m in peaks:
    print(f"  {f:7.1f} Hz   magnitude {m:.5f}")

plot_spectrum(freqs, mag, phase, peaks, RESULTS_DIR / "fft.png");

The two strongest peaks fall near 293.6 Hz and 393.8 Hz, close to D4 (293.7 Hz) and G4 (392.0 Hz), the notes common to both chords. Other detected peaks line up with the remaining chord tones (for example about 131 Hz near C3 and 197 Hz near G3) and with harmonics such as about 587 Hz, the second harmonic of D4.

Phase is only drawn for bins within 30 dB of the peak magnitude; below that, the phase of a near-zero bin is numerically meaningless. Even for significant bins, the phase of a long clip with several strum onsets has little visible structure, so the magnitude spectrum and spectrogram carry most of the useful information here.

In [ ]:
plot_spectrogram(x, fs, RESULTS_DIR / "spectrogram.png");

## Audio processing

The effects chain has two stages: soft clipping first, then EQ.

### Nonlinear saturation: tanh soft clipping

$$y[n] = \frac{\tanh(\text{drive} \cdot x[n])}{\tanh(\text{drive})}$$

The tanh curve compresses large samples smoothly instead of cutting them off, giving an overdrive-style saturation. It is a simple static nonlinearity that approximates the character of overdrive; it is not a physical model of a tube amplifier stage. Because it is nonlinear, it creates frequency components that were not in the input: harmonics of each note and, since a chord contains several notes at once, intermodulation products. A drive of 8 was chosen by ear; much higher values approach hard clipping and sound buzzy.

### Frequency-selective EQ: Butterworth filters

Three Butterworth stages reshape the spectrum:

- **High-pass at 100 Hz** (4th order): reduces low-frequency body resonance.
- **Low-pass at 6 kHz** (4th order): rolls off high-frequency acoustic "air".
- **Presence boost at 2-4 kHz**: a 2nd-order band-pass copy of the signal is added back to it. Near the band center this roughly doubles the in-band amplitude, about +6 dB.

Each stage is applied with `scipy.signal.filtfilt`, which runs the filter forward and then backward. The result has zero phase distortion and a squared magnitude response, but it needs the whole signal in advance, so this is an offline, non-causal pipeline rather than a real-time effect.

After the EQ, the signal is peak-normalized and scaled by 0.95 to leave headroom before conversion to 16-bit PCM.

In [ ]:
clipped = peak_normalize(soft_clip(x, DRIVE))
processed = apply_guitar_eq(clipped, fs) * HEADROOM

save_audio(PROCESSED_WAV, processed, fs)
print(f"Soft clip drive {DRIVE}, EQ: HPF 100 Hz, LPF 6 kHz, 2-4 kHz presence, gain {HEADROOM}")

print("Source")
display(Audio(x, rate=fs))
print("Processed")
display(Audio(processed, rate=fs))

## Result comparison

Waveform, spectrum, and spectrogram comparisons of the source and processed signals.

In [ ]:
generate_comparisons(t, x, processed, fs, RESULTS_DIR)

In the time domain, the processed signal has a flatter, more sustained envelope because soft clipping compresses the strum transients. In the spectrogram, the processed signal shows dense new energy from roughly 1 kHz up to the 6 kHz low-pass, produced by the nonlinearity rather than by the filters, since linear filters can only rescale components that already exist. The high-pass removes most of the low-frequency content below 100 Hz.

## Reflection

| Technique | Role |
|-----------|------|
| `wavfile.read` and stereo-to-mono averaging | Load and condition the real recording |
| FFT (`scipy.fft.rfft`) | Frequency content of the clip; identify chord tones |
| STFT spectrogram | Time-varying harmonic content |
| tanh soft clipping | Nonlinear saturation that adds harmonics |
| Butterworth high-pass, low-pass, band-pass | LTI frequency-selective shaping |

**Linear vs. nonlinear.** Each Butterworth stage is an LTI system with a rational transfer function H(z); cascading them multiplies their frequency responses. The tanh clipper is not LTI: it changes the waveform shape and generates new frequency content. For a single sinusoid, a symmetric nonlinearity like tanh produces mainly odd harmonics; for a chord it also produces sum and difference products between notes.

**Normalization.** Adding the presence band back pushes the peak amplitude above 1.0, so the signal is re-normalized after EQ, and the final 0.95 gain keeps the exported 16-bit samples away from full scale.

**Presence band.** The 2-4 kHz region is where hearing is most sensitive, and boosting it is a common way to help a guitar cut through a mix.

**Chord verification from the spectrum.** Matching the dominant FFT peaks against standard tuning confirms which notes were played, which was a useful sanity check that the analysis matched what I actually recorded.